# 02. Model Training & Evaluation - Teen Mental Health Prediction

This notebook covers data preprocessing, feature encoding, train-test splitting, SMOTE class balancing, model training, evaluation on untouched test data, and model saving.

### Workflow Steps:
1. Load raw dataset (`data/raw/teen_mental_health.csv`)
2. Separate features ($X$) and target ($y = \text{depression\_label}$)
3. Perform Stratified Train-Test Split (80/20, `random_state=42`)
4. Apply **SMOTE** to **TRAINING set ONLY** to handle severe class imbalance
5. Fit **StandardScaler** on training data and transform train & test sets
6. Train three classification models (Logistic Regression, Random Forest, Gradient Boosting)
7. Evaluate on untouched test set using Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices
8. Compare all models in a unified summary table
9. Plot overlaid ROC curves for performance comparison
10. Save best model (`models/best_model.pkl`), scaler (`models/scaler.pkl`), and feature schema

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, classification_report
)
from imblearn.over_sampling import SMOTE

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11

## 1. Load Data & Feature Preprocessing

In [ ]:
# Determine dataset path relative to execution context
data_path = '../data/raw/teen_mental_health.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/teen_mental_health.csv'

df = pd.read_csv(data_path)
print(f"Dataset loaded. Shape: {df.shape}")

# Separate Target (y) and Features (X)
target_col = 'depression_label'
X_raw = df.drop(columns=[target_col])
y = df[target_col]

# One-Hot Encode categorical features
X = pd.get_dummies(X_raw, drop_first=True, dtype=int)
feature_names = list(X.columns)

print(f"\nFeatures shape after encoding: {X.shape}")
print(f"Encoded features ({len(feature_names)}): {feature_names}")
print(f"\nTarget distribution:\n{y.value_counts(normalize=True).map('{:.2%}'.format)}")

## 2. Stratified Train-Test Split & SMOTE Balancing

In [ ]:
# 1. Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("=== Class Distribution Before SMOTE ===")
print(f"Train set (y_train): {dict(y_train.value_counts())}")
print(f"Test set (y_test):   {dict(y_test.value_counts())}")

# 2. Apply SMOTE to TRAINING SET ONLY (Never touch the test set)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\n=== Class Distribution After SMOTE (Training Set Only) ===")
print(f"Resampled y_train:   {dict(y_train_res.value_counts())}")
print(f"Test set remains untouched: {dict(y_test.value_counts())}")

## 3. Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()

# Fit scaler on training set (resampled), transform both train and test sets
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully with StandardScaler.")

## 4. Model Training & Evaluation

In [ ]:
# Initialize three models for comparison
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42, n_estimators=100)
}

results = {}
prob_predictions = {}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for idx, (name, model) in enumerate(models.items()):
    # Train model on scaled SMOTE training data
    model.fit(X_train_scaled, y_train_res)
    
    # Predict on untouched test set
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    prob_predictions[name] = y_prob
    
    # Calculate metrics
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        "ModelObject": model,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    }
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-Depressed (0)", "Depressed (1)"])
    disp.plot(ax=axes[idx], cmap="Blues", cbar=False)
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Model Metrics Comparison Table

In [ ]:
metrics_data = {}
for name, res in results.items():
    metrics_data[name] = {
        "Precision": res["Precision"],
        "Recall": res["Recall"],
        "F1-Score": res["F1-Score"],
        "ROC-AUC": res["ROC-AUC"]
    }

comparison_df = pd.DataFrame(metrics_data).T.round(4)
print("=== Model Evaluation Comparison (Untouched Test Set) ===")
display(comparison_df)

## 6. ROC Curves Comparison

In [ ]:
plt.figure(figsize=(8, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for (name, y_prob), color in zip(prob_predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = results[name]["ROC-AUC"]
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label="Random Baseline (AUC = 0.500)")
plt.xlabel("False Positive Rate", fontsize=11)
plt.ylabel("True Positive Rate (Recall)", fontsize=11)
plt.title("ROC Curves Comparison on Untouched Test Set", fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Model Selection & Persistence

In [ ]:
# Determine best model based on F1-Score for minority class
best_model_name = comparison_df['F1-Score'].idxmax()
best_model = results[best_model_name]["ModelObject"]

print(f"🏆 Best Model Selected (Highest F1-Score): {best_model_name}")
print(f"   - F1-Score: {comparison_df.loc[best_model_name, 'F1-Score']}")
print(f"   - ROC-AUC:  {comparison_df.loc[best_model_name, 'ROC-AUC']}")

# Ensure models directory exists
output_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'best_model.pkl')
scaler_path = os.path.join(output_dir, 'scaler.pkl')
features_path = os.path.join(output_dir, 'feature_names.pkl')

# Save model, scaler, and encoded feature schema
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, features_path)

print(f"\nSaved artifacts to '{output_dir}/':")
print(f"  - Best Model:    {model_path}")
print(f"  - Scaler:        {scaler_path}")
print(f"  - Feature Names: {features_path}")